# District Health Risk Score
## BenefitBridge AI — Track 2: Medical Desert Planner

**What this notebook does:**
1. Loads `district_health_context` from Unity Catalog
2. Computes raw sub-scores for five clinical domains
3. Applies MinMaxScaler normalisation across all 706 districts
4. Computes weighted composite `district_risk_score` (0–100)
5. Assigns `risk_tier` and `dominant_domain`
6. Writes `trusted.district_risk_scores` back to Delta
7. Joins with facility density to produce `trusted.medical_desert_index`

**Model:** Weighted Domain Composite Index (UNDP HDI methodology)
**Source:** `benefits_navigator.trusted.district_health_context` — 706 districts


## 0 · Setup


In [0]:
# ================================================================
# Fix: State name mismatches between NFHS and facilities data
# Run this as a NEW notebook cell BEFORE re-running the risk score
#
# Problem: NFHS uses different spellings than facilities data
#   NFHS says    → facilities say
#   MAHARASTRA   → MAHARASHTRA        (missing H)
#   JAMMU & KASHMIR → JAMMU AND KASHMIR
#   ANDAMAN & NICOBAR ISLANDS → ANDAMAN AND NICOBAR ISLANDS
#   DADRA AND NAGAR HAVELI & DAMAN AND DIU → THE DADRA...
#
# Fix: patch district_health_context to match facilities spelling
# Then re-run cells 3-13 of the risk score notebook
# ================================================================

# ── STEP 1: Check current row count (should be 706) ─────────────
before = spark.table(tbl("district_health_context")).count()
print(f"Rows before fix: {before}")

# ── STEP 2: Patch state_norm in district_health_context ─────────
spark.sql(f"""
    CREATE OR REPLACE TABLE {tbl('district_health_context')}
    USING DELTA AS
    SELECT
        * EXCEPT(state_norm),
        CASE state_norm
            WHEN 'MAHARASTRA'
                THEN 'MAHARASHTRA'
            WHEN 'JAMMU & KASHMIR'
                THEN 'JAMMU AND KASHMIR'
            WHEN 'ANDAMAN & NICOBAR ISLANDS'
                THEN 'ANDAMAN AND NICOBAR ISLANDS'
            WHEN 'DADRA AND NAGAR HAVELI & DAMAN AND DIU'
                THEN 'THE DADRA AND NAGAR HAVELI AND DAMAN AND DIU'
            ELSE state_norm
        END AS state_norm
    FROM {tbl('district_health_context')}
""")

# ── STEP 3: Verify row count unchanged ──────────────────────────
after = spark.table(tbl("district_health_context")).count()
print(f"Rows after fix : {after}")
assert before == after, f"Row count changed! {before} → {after}"
print("Row count unchanged ✓")

# ── STEP 4: Confirm Maharashtra now exists ───────────────────────
spark.sql(f"""
    SELECT DISTINCT state_norm
    FROM {tbl('district_health_context')}
    WHERE state_norm IN (
        'MAHARASHTRA',
        'JAMMU AND KASHMIR',
        'ANDAMAN AND NICOBAR ISLANDS',
        'THE DADRA AND NAGAR HAVELI AND DAMAN AND DIU'
    )
    ORDER BY state_norm
""").show(truncate=False)

# ── STEP 5: Show Maharashtra districts as sanity check ───────────
count = spark.sql(f"""
    SELECT COUNT(*) AS n
    FROM {tbl('district_health_context')}
    WHERE state_norm = 'MAHARASHTRA'
""").collect()[0]['n']
print(f"Maharashtra districts in NFHS: {count}  (expect 36)")

In [0]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, StringType, TimestampType
from datetime import datetime

CATALOG = "benefits_navigator"
SCHEMA  = "trusted"

def tbl(name):
    return f"`{CATALOG}`.`{SCHEMA}`.`{name}`"

print("Libraries loaded.")


## 1 · Load district_health_context


In [0]:
df_raw = spark.table(tbl("district_health_context")).toPandas()

print(f"Rows loaded : {len(df_raw)}")
print(f"Columns     : {len(df_raw.columns)}")
print(f"Sample districts:\n{df_raw[['district_norm','state_norm']].head(5).to_string(index=False)}")


## 2 · Domain indicator definitions

Each domain is a dict of `{column: direction}` where:
- `+1`  = higher value = MORE at-risk (bad outcome)
- `-1`  = higher value = LESS at-risk (good outcome, inverted before scoring)


In [0]:
DOMAINS = {

    # ── D1: Maternal Health (weight 25%) ─────────────────────────────
    # High-weight domain: maternal mortality proxy indicators
    "maternal": {
        "columns": {
            "anc_4visits_pct":         {"direction": -1, "weight": 0.30},  # LOW = bad
            "institutional_birth_pct": {"direction": -1, "weight": 0.30},  # LOW = bad
            "pnc_skilled_pct":         {"direction": -1, "weight": 0.20},  # LOW = bad
            "ifa_180d_pct":            {"direction": -1, "weight": 0.10},  # LOW = bad
            "teen_pregnancy_pct":      {"direction": +1, "weight": 0.10},  # HIGH = bad
        },
        "domain_weight": 0.25,
    },

    # ── D2: Child Health (weight 25%) ────────────────────────────────
    # Stunting and wasting are the most sensitive chronic deprivation indicators
    "child": {
        "columns": {
            "stunting_pct":           {"direction": +1, "weight": 0.25},  # HIGH = bad
            "wasting_pct":            {"direction": +1, "weight": 0.20},  # HIGH = bad
            "underweight_pct":        {"direction": +1, "weight": 0.20},  # HIGH = bad
            "child_anaemia_pct":      {"direction": +1, "weight": 0.15},  # HIGH = bad
            "fully_vaccinated_pct":   {"direction": -1, "weight": 0.15},  # LOW  = bad
            "diarrhoea_2wk_pct":      {"direction": +1, "weight": 0.05},  # HIGH = bad
        },
        "domain_weight": 0.25,
    },

    # ── D3: Healthcare Access (weight 25%) ───────────────────────────
    # Structural access determines whether burden can be addressed
    "access": {
        "columns": {
            "health_insurance_pct":     {"direction": -1, "weight": 0.20},  # LOW  = bad
            "fp_unmet_need_pct":        {"direction": +1, "weight": 0.20},  # HIGH = bad
            "women_literate_pct":       {"direction": -1, "weight": 0.20},  # LOW  = bad
            "improved_water_pct":       {"direction": -1, "weight": 0.15},  # LOW  = bad
            "improved_sanitation_pct":  {"direction": -1, "weight": 0.15},  # LOW  = bad
            "clean_fuel_pct":           {"direction": -1, "weight": 0.10},  # LOW  = bad
        },
        "domain_weight": 0.25,
    },

    # ── D4: Women's Health (weight 15%) ──────────────────────────────
    # Gender-specific burden underweighted in general indices
    "womens": {
        "columns": {
            "women_anaemia_pct":          {"direction": +1, "weight": 0.30},  # HIGH = bad
            "cervical_screen_pct":        {"direction": -1, "weight": 0.25},  # LOW  = bad
            "breast_exam_pct":            {"direction": -1, "weight": 0.20},  # LOW  = bad
            "women_high_bp_pct":          {"direction": +1, "weight": 0.15},  # HIGH = bad
            "women_high_blood_sugar_pct": {"direction": +1, "weight": 0.10},  # HIGH = bad
        },
        "domain_weight": 0.15,
    },

    # ── D5: Child Immunisation (weight 10%) ──────────────────────────
    # Most actionable preventive metric; proxy for facility reach
    # Note: fully_vaccinated_pct already in D2 — use component vaccines here
    "immunisation": {
        "columns": {
            "bcg_pct":    {"direction": -1, "weight": 0.35},  # LOW = bad
            "polio3_pct": {"direction": -1, "weight": 0.35},  # LOW = bad
            "dpt3_pct":   {"direction": -1, "weight": 0.30},  # LOW = bad
        },
        "domain_weight": 0.10,
    },
}

# Verify domain weights sum to 1.0
total_domain_weight = sum(d["domain_weight"] for d in DOMAINS.values())
assert abs(total_domain_weight - 1.0) < 1e-9, f"Domain weights sum to {total_domain_weight}, not 1.0"
print(f"Domain weights validated: {total_domain_weight:.2f}")


## 3 · Compute raw domain sub-scores

For each domain:
- Invert negative-direction indicators (good outcome → risk contribution)
- Apply per-indicator weights within the domain
- Handle NULLs: if ≥ 3 indicators missing → domain score = NaN (flagged later)
- Raw score range: 0–100 (not yet normalised across districts)


In [0]:
def compute_raw_domain_score(row, domain_config):
    """
    Compute weighted raw score for one district on one domain.
    Returns (raw_score, null_count, available_weight_sum).
    """
    cols     = domain_config["columns"]
    weighted = 0.0
    null_count        = 0
    available_weight  = 0.0

    for col, cfg in cols.items():
        val = row.get(col)
        if pd.isna(val):
            null_count += 1
            continue

        # Invert if direction == -1 (higher value = better = lower risk)
        risk_contribution = (100.0 - float(val)) if cfg["direction"] == -1 else float(val)

        weighted         += risk_contribution * cfg["weight"]
        available_weight += cfg["weight"]

    # NULL policy: if fewer than 3 indicators available → NaN
    total_indicators = len(cols)
    if (total_indicators - null_count) < max(2, total_indicators - 2):
        return np.nan, null_count, 0.0

    # Rescale by available weight so partial data doesn't compress scores
    if available_weight > 0:
        raw_score = weighted / available_weight * 100.0
    else:
        raw_score = np.nan

    return raw_score, null_count, available_weight


# Apply to every district
raw_scores = {f"{name}_raw": [] for name in DOMAINS}
null_counts = {f"{name}_null_count": [] for name in DOMAINS}

for _, row in df_raw.iterrows():
    for domain_name, domain_config in DOMAINS.items():
        raw, nulls, _ = compute_raw_domain_score(row, domain_config)
        raw_scores[f"{domain_name}_raw"].append(raw)
        null_counts[f"{domain_name}_null_count"].append(nulls)

df_raw_scores = df_raw[["district_norm", "state_norm"]].copy()
for col, vals in {**raw_scores, **null_counts}.items():
    df_raw_scores[col] = vals

print("Raw domain scores computed.")
print(df_raw_scores[[c for c in df_raw_scores if c.endswith("_raw")]].describe().round(1))


## 4 · MinMaxScaler normalisation (the ML step)

Each domain's raw scores are scaled across all 706 districts:
- Lowest-burden district → 0
- Highest-burden district → 100
- NaN districts are excluded from scaler fit but assigned NaN in output

This is equivalent to `sklearn.preprocessing.MinMaxScaler`
applied independently per domain column.


In [0]:
scaler = MinMaxScaler(feature_range=(0, 100))

domain_raw_cols    = [f"{name}_raw"   for name in DOMAINS]
domain_score_cols  = [f"{name}_score" for name in DOMAINS]

df_scores = df_raw_scores.copy()

for raw_col, score_col in zip(domain_raw_cols, domain_score_cols):
    col_data = df_scores[[raw_col]].copy()
    mask_valid = col_data[raw_col].notna()

    if mask_valid.sum() < 2:
        df_scores[score_col] = np.nan
        print(f"  {raw_col}: insufficient data for scaling → all NaN")
        continue

    # Fit scaler on non-NaN values only
    scaler_single = MinMaxScaler(feature_range=(0, 100))
    scaler_single.fit(col_data.loc[mask_valid, [raw_col]])

    # Transform all rows; NaN rows stay NaN
    scaled = np.full(len(col_data), np.nan)
    scaled[mask_valid.values] = scaler_single.transform(
        col_data.loc[mask_valid, [raw_col]]
    ).flatten()

    df_scores[score_col] = np.round(scaled, 1)
    print(f"  {raw_col}: min={df_scores[score_col].min():.1f}  "
          f"max={df_scores[score_col].max():.1f}  "
          f"mean={df_scores[score_col].mean():.1f}  "
          f"nulls={df_scores[score_col].isna().sum()}")

print("\nNormalised domain scores ready.")


## 5 · Weighted composite district_risk_score

Composite = weighted average of available normalised domain scores.
If a domain score is NaN, its weight is redistributed proportionally
to the available domains so the total always sums to 1.0.


In [0]:
def compute_composite(row, domains):
    """
    Weighted composite from normalised domain scores.
    Redistributes weight of NaN domains to available ones.
    Returns (composite_score, data_completeness_pct, domains_available)
    """
    available = {}
    for name, cfg in domains.items():
        val = row.get(f"{name}_score")
        if not pd.isna(val):
            available[name] = (val, cfg["domain_weight"])

    if not available:
        return np.nan, 0.0, []

    total_available_weight = sum(w for _, w in available.values())
    composite = sum(
        val * (w / total_available_weight)
        for val, w in available.values()
    )
    completeness = len(available) / len(domains) * 100.0
    return round(composite, 1), round(completeness, 1), list(available.keys())


composites      = []
completeness_pcts = []
domains_used_list = []

for _, row in df_scores.iterrows():
    comp, compl, used = compute_composite(row, DOMAINS)
    composites.append(comp)
    completeness_pcts.append(compl)
    domains_used_list.append(", ".join(used))

df_scores["district_risk_score"]   = composites
df_scores["data_completeness_pct"] = completeness_pcts
df_scores["domains_available"]     = domains_used_list

print("Composite scores computed.")
print(df_scores["district_risk_score"].describe().round(1))


## 6 · Risk tier and dominant domain


In [0]:
def assign_tier(score):
    if pd.isna(score): return "unscored"
    if score >= 75:    return "critical"
    if score >= 55:    return "high"
    if score >= 35:    return "moderate"
    return "low"

def dominant_domain(row, domains):
    """Which normalised domain score is highest for this district?"""
    best_domain = None
    best_score  = -1.0
    for name in domains:
        val = row.get(f"{name}_score")
        if not pd.isna(val) and val > best_score:
            best_score  = val
            best_domain = name
    return best_domain

df_scores["risk_tier"]       = df_scores["district_risk_score"].apply(assign_tier)
df_scores["dominant_domain"] = df_scores.apply(
    lambda r: dominant_domain(r, DOMAINS), axis=1
)

print("\nRisk tier distribution:")
print(df_scores["risk_tier"].value_counts().to_string())
print("\nDominant domain distribution:")
print(df_scores["dominant_domain"].value_counts().to_string())


## 7 · Preview top and bottom 10 districts


In [0]:
display_cols = [
    "district_norm", "state_norm",
    "district_risk_score", "risk_tier", "dominant_domain",
    "maternal_score", "child_score", "access_score",
    "womens_score", "immunisation_score",
    "data_completeness_pct"
]

print("── TOP 10 HIGHEST RISK ──")
print(
    df_scores[display_cols]
    .sort_values("district_risk_score", ascending=False)
    .head(10)
    .to_string(index=False)
)

print("\n── TOP 10 LOWEST RISK ──")
print(
    df_scores[display_cols]
    .sort_values("district_risk_score", ascending=True)
    .head(10)
    .to_string(index=False)
)


## 8 · Write district_risk_scores to Delta


In [0]:
output_cols = [
    "district_norm",
    "state_norm",
    "district_risk_score",
    "risk_tier",
    "dominant_domain",
    "maternal_score",
    "child_score",
    "access_score",
    "womens_score",
    "immunisation_score",
    "data_completeness_pct",
    "domains_available",
    # Raw scores kept for audit / explainability
    "maternal_raw",
    "child_raw",
    "access_raw",
    "womens_raw",
    "immunisation_raw",
    # NULL counts per domain
    "maternal_null_count",
    "child_null_count",
    "access_null_count",
    "womens_null_count",
    "immunisation_null_count",
]

df_out = df_scores[output_cols].copy()
df_out["scored_at"] = datetime.utcnow()

sdf_risk = spark.createDataFrame(df_out)

(
    sdf_risk.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tbl("district_risk_scores"))
)

count = spark.table(tbl("district_risk_scores")).count()
print(f"Written {count} rows to {tbl('district_risk_scores')}")


## 9 · Facility density per district

Count verified facilities (geo_match_method != 'no_zip') per district.
This is the supply-side input to the medical desert index.


In [0]:
sdf_facility_density = spark.sql(f"""
    SELECT
        fe.district_norm,
        fe.state_norm,
        COUNT(*)                                            AS total_facilities,
        SUM(CASE WHEN fe.geo_match_method != 'no_zip'
                  AND fe.is_dark_facility = false
                 THEN 1 ELSE 0 END)                        AS verified_facilities,
        SUM(CASE WHEN fe.facility_type LIKE '%hospital%'
                  OR fe.facility_type LIKE '%Hospital%'
                 THEN 1 ELSE 0 END)                        AS hospital_count,
        SUM(CASE WHEN fe.operator_type = 'public'
                  OR fe.operator_type LIKE '%govt%'
                  OR fe.operator_type LIKE '%Govt%'
                 THEN 1 ELSE 0 END)                        AS public_facilities,
        ROUND(AVG(ft.trust_score), 1)                      AS avg_trust_score
    FROM {tbl('facilities_enriched')} fe
    LEFT JOIN {tbl('facility_trust_scores')} ft
        ON fe.unique_id = ft.unique_id
    WHERE fe.district_norm IS NOT NULL
    GROUP BY fe.district_norm, fe.state_norm
""")

print(f"Facility density computed for {sdf_facility_density.count()} districts.")
sdf_facility_density.orderBy(F.desc("verified_facilities")).show(10, truncate=False)

## 10 · Medical Desert Index

**Formula:**
```
medical_desert_index = district_risk_score × (1 / LOG2(verified_facilities + 2))
```

Using LOG base 2 with +2 offset so:
- 0 facilities: divisor = LOG2(2) = 1.0  → full risk score preserved
- 1 facility:   divisor = LOG2(3) ≈ 1.58 → slight dampening
- 10 facilities: divisor = LOG2(12) ≈ 3.6 → meaningful reduction
- 100+ facilities: high divisor → low desert index despite high risk

Higher desert_index = higher burden AND fewer facilities = true medical desert.


In [0]:
sdf_risk_spark = spark.table(tbl("district_risk_scores"))

sdf_desert = (
    sdf_risk_spark.alias("r")
    .join(
        sdf_facility_density.alias("f"),
        on=["district_norm", "state_norm"],
        how="left"
    )
    .withColumn(
        "verified_facilities",
        F.coalesce(F.col("f.verified_facilities"), F.lit(0))
    )
    .withColumn(
        "total_facilities",
        F.coalesce(F.col("f.total_facilities"), F.lit(0))
    )
    # Medical desert index: risk × inverse log facility density
    .withColumn(
        "medical_desert_index",
        F.when(
            F.col("district_risk_score").isNull(), None
        ).otherwise(
            F.round(
                F.col("district_risk_score") /
                F.log2(F.col("verified_facilities") + 2),
                1
            )
        )
    )
    # Desert tier: top quintile = true desert
    .withColumn(
        "desert_tier",
        F.when(F.col("medical_desert_index").isNull(), "unscored")
         .when(F.col("medical_desert_index") >= 60,    "desert")
         .when(F.col("medical_desert_index") >= 40,    "underserved")
         .when(F.col("medical_desert_index") >= 20,    "partial")
         .otherwise(                                    "served")
    )
    # Facility gap flag: high risk + very few verified facilities
    .withColumn(
        "facility_gap_flag",
        F.when(
            (F.col("district_risk_score") >= 55) &
            (F.col("verified_facilities") < 5),
            True
        ).otherwise(False)
    )
    .select(
        "r.district_norm",
        "r.state_norm",
        "r.district_risk_score",
        "r.risk_tier",
        "r.dominant_domain",
        "r.maternal_score",
        "r.child_score",
        "r.access_score",
        "r.womens_score",
        "r.immunisation_score",
        "r.data_completeness_pct",
        "verified_facilities",
        "total_facilities",
        F.col("f.hospital_count"),
        F.col("f.public_facilities"),
        F.col("f.avg_trust_score"),
        "medical_desert_index",
        "desert_tier",
        "facility_gap_flag",
        F.lit(datetime.utcnow()).alias("scored_at"),
    )
)

print("Medical desert index computed.")
print("\nDesert tier distribution:")
sdf_desert.groupBy("desert_tier").count().orderBy("desert_tier").show()

print("\nTop 15 medical deserts:")
sdf_desert.filter(F.col("medical_desert_index").isNotNull()) \
    .orderBy(F.desc("medical_desert_index")) \
    .select("district_norm","state_norm","district_risk_score",
            "verified_facilities","medical_desert_index","desert_tier") \
    .show(15, truncate=False)


## 11 · Write medical_desert_index to Delta


In [0]:
(
    sdf_desert.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tbl("medical_desert_index"))
)

count = spark.table(tbl("medical_desert_index")).count()
print(f"Written {count} rows to {tbl('medical_desert_index')}")


## 12 · Validation summary


In [0]:
spark.sql(f"""
    SELECT
        risk_tier,
        desert_tier,
        COUNT(*)                              AS districts,
        ROUND(AVG(district_risk_score), 1)    AS avg_risk,
        ROUND(AVG(verified_facilities), 1)    AS avg_facilities,
        ROUND(AVG(medical_desert_index), 1)   AS avg_desert_idx,
        SUM(CAST(facility_gap_flag AS INT))   AS gap_flagged
    FROM {tbl('medical_desert_index')}
    GROUP BY risk_tier, desert_tier
    ORDER BY avg_risk DESC, avg_desert_idx DESC
""").show(20, truncate=False)


## 13 · Explainability check

For any district, show the exact indicator values that drove the score.
This is what judges and planners can audit.


In [0]:
SAMPLE_DISTRICT = "WASHIM"   # change to any district in your data
SAMPLE_STATE    = "MAHARASHTRA"

spark.sql(f"""
    SELECT
        r.district_norm,
        r.state_norm,
        r.district_risk_score,
        r.risk_tier,
        r.dominant_domain,
        r.maternal_score,   r.maternal_raw,
        r.child_score,      r.child_raw,
        r.access_score,     r.access_raw,
        r.womens_score,     r.womens_raw,
        r.immunisation_score, r.immunisation_raw,
        -- Raw NFHS indicators
        d.anc_4visits_pct,
        d.institutional_birth_pct,
        d.stunting_pct,
        d.fully_vaccinated_pct,
        d.health_insurance_pct,
        d.women_anaemia_pct,
        d.fp_unmet_need_pct
    FROM {tbl('district_risk_scores')} r
    JOIN {tbl('district_health_context')} d
        ON  r.district_norm = d.district_norm
        AND r.state_norm    = d.state_norm
    WHERE r.district_norm = '{SAMPLE_DISTRICT}'
      AND r.state_norm    = '{SAMPLE_STATE}'
""").show(1, vertical=True, truncate=False)


In [0]:
# Check district_risk_scores directly
print("=== district_risk_scores ===")
spark.sql(f"""
    SELECT district_norm, state_norm, district_risk_score, risk_tier
    FROM {tbl('district_risk_scores')}
    WHERE state_norm LIKE '%MAHARASHTRA%'
    ORDER BY district_risk_score DESC
    LIMIT 10
""").show(truncate=False)

# Check district_health_context directly  
print("=== district_health_context ===")
spark.sql(f"""
    SELECT district_norm, state_norm
    FROM {tbl('district_health_context')}
    WHERE state_norm LIKE '%MAHARASHTRA%'
    ORDER BY district_norm
    LIMIT 10
""").show(truncate=False)

In [0]:
# Find what NFHS actually calls Maharashtra
spark.sql(f"""
    SELECT DISTINCT state_norm
    FROM {tbl('district_health_context')}
    ORDER BY state_norm
""").show(100, truncate=False)

In [0]:
# ================================================================
# BenefitBridge AI — District Risk Map Visualisation
# Add these cells to your district_risk_score notebook
# Run AFTER cell 13 (after medical_desert_index is written to Delta)
# ================================================================


# ── CELL 1 of 6: Install libraries ──────────────────────────────
# Run this first, restart kernel if prompted

%pip install plotly geopandas requests --quiet


# ── CELL 2 of 6: Imports + load data ────────────────────────────

import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import numpy as np
import requests, json, os

# Load medical_desert_index from Delta
# tbl() is already defined earlier in the notebook
df_map = spark.table(tbl("medical_desert_index")).toPandas()

print(f"Districts loaded : {len(df_map)}")
print(f"\nDesert tier distribution:")
print(df_map['desert_tier'].value_counts().to_string())
print(f"\nDesert index range: "
      f"{df_map['medical_desert_index'].min():.1f} "
      f"– {df_map['medical_desert_index'].max():.1f}")


# ── CELL 3 of 6: Fetch India district GeoJSON ───────────────────
# Public domain, Census 2011, ~640 district polygons
# Falls back to state-level chart if fetch fails

GEOJSON_URL = (
    "https://raw.githubusercontent.com/datameet/maps/master/"
    "Districts/Census_2011/2011_Dist.geojson"
)

try:
    resp = requests.get(GEOJSON_URL, timeout=30)
    resp.raise_for_status()
    india_geojson = resp.json()
    print(f"GeoJSON loaded: {len(india_geojson['features'])} polygons")

    # Show available property keys so you know which to join on
    sample_props = india_geojson['features'][0]['properties']
    print(f"Properties: {list(sample_props.keys())}")
    print(f"Sample    : {sample_props}")
    GEOJSON_OK = True

except Exception as e:
    print(f"GeoJSON fetch failed: {e}")
    print("Falling back to state-level bar chart")
    GEOJSON_OK = False


# ── CELL 4 of 6: District choropleth (Option A) ─────────────────
# Skipped automatically if GeoJSON unavailable

if GEOJSON_OK:

    # Auto-detect property key names in the GeoJSON
    df_geo_props = pd.DataFrame(
        [f['properties'] for f in india_geojson['features']]
    )
    DISTRICT_COL, STATE_COL = None, None
    for col in df_geo_props.columns:
        vals = df_geo_props[col].dropna().astype(str).str.upper().tolist()[:10]
        if any(v in ['PUNE', 'MUMBAI', 'DELHI', 'LUCKNOW', 'PATNA']
               for v in vals):
            DISTRICT_COL = col
        if any(v in ['MAHARASHTRA', 'KARNATAKA', 'UTTAR PRADESH', 'BIHAR']
               for v in vals):
            STATE_COL = col

    print(f"District column : {DISTRICT_COL}")
    print(f"State column    : {STATE_COL}")

    # Build composite join key on GeoJSON features
    for feat in india_geojson['features']:
        p = feat['properties']
        d = str(p.get(DISTRICT_COL, '')).upper().strip()
        s = str(p.get(STATE_COL, '')).upper().strip()
        feat['properties']['join_key'] = f"{d}||{s}"

    # Build same key on our data
    df_map['join_key'] = (
        df_map['district_norm'].str.upper().str.strip() + '||' +
        df_map['state_norm'].str.upper().str.strip()
    )

    geo_keys  = {f['properties']['join_key'] for f in india_geojson['features']}
    data_keys = set(df_map['join_key'])
    matched   = geo_keys & data_keys
    print(f"\nGeoJSON polygons : {len(geo_keys)}")
    print(f"Data districts   : {len(data_keys)}")
    print(f"Matched          : {len(matched)} "
          f"({len(matched)/len(data_keys)*100:.1f}% of data)")

    # Colour scale: green → yellow → orange → red
    COLOR_SCALE = [
        [0.0, '#2d6a4f'],   # served       — dark green
        [0.3, '#74c69d'],   # partial low  — light green
        [0.5, '#ffd166'],   # partial high — yellow
        [0.7, '#ef6c00'],   # underserved  — orange
        [1.0, '#b91c1c'],   # desert       — dark red
    ]

    fig_district = px.choropleth(
        df_map,
        geojson=india_geojson,
        locations='join_key',
        featureidkey='properties.join_key',
        color='medical_desert_index',
        color_continuous_scale=COLOR_SCALE,
        range_color=[0, df_map['medical_desert_index'].quantile(0.98)],
        hover_name='district_norm',
        hover_data={
            'state_norm':           True,
            'district_risk_score':  ':.1f',
            'medical_desert_index': ':.1f',
            'verified_facilities':  True,
            'desert_tier':          True,
            'dominant_domain':      True,
            'join_key':             False,
        },
        title='India Medical Desert Index by District',
        labels={
            'medical_desert_index': 'Desert Index',
            'district_risk_score':  'Risk Score',
            'verified_facilities':  'Verified Facilities',
            'state_norm':           'State',
            'desert_tier':          'Tier',
            'dominant_domain':      'Primary Gap',
        },
    )
    fig_district.update_geos(
        fitbounds='locations',
        visible=False,
        bgcolor='rgba(0,0,0,0)',
    )
    fig_district.update_layout(
        title_font_size=18,
        title_x=0.5,
        margin=dict(l=0, r=0, t=50, b=0),
        height=650,
        coloraxis_colorbar=dict(
            title='Desert Index',
            tickvals=[0, 20, 40, 60, 80],
            ticktext=['0\nServed', '20', '40\nPartial', '60\nDesert', '80+'],
            len=0.7,
        ),
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
    )
    displayHTML(fig_district.to_html(full_html=False, include_plotlyjs='cdn'))


# ── CELL 5 of 6: State bar chart + scatter (always runs) ────────

# ── State rollup ──────────────────────────────────────────────
df_state = (
    df_map.groupby('state_norm')
    .agg(
        avg_desert_index  = ('medical_desert_index', 'mean'),
        avg_risk_score    = ('district_risk_score',  'mean'),
        total_districts   = ('district_norm',        'count'),
        desert_districts  = ('desert_tier',
                             lambda x: (x == 'desert').sum()),
        critical_districts= ('risk_tier',
                             lambda x: (x == 'critical').sum()),
        total_facilities  = ('verified_facilities',  'sum'),
    )
    .reset_index()
)
df_state['avg_desert_index'] = df_state['avg_desert_index'].round(1)
df_state['avg_risk_score']   = df_state['avg_risk_score'].round(1)
df_state['desert_pct'] = (
    df_state['desert_districts'] / df_state['total_districts'] * 100
).round(1)

fig_state = px.bar(
    df_state.sort_values('avg_desert_index', ascending=True).tail(20),
    x='avg_desert_index',
    y='state_norm',
    orientation='h',
    color='avg_desert_index',
    color_continuous_scale=[
        [0.0, '#2d6a4f'],
        [0.4, '#ffd166'],
        [1.0, '#b91c1c'],
    ],
    hover_data={
        'total_districts':    True,
        'desert_districts':   True,
        'desert_pct':         True,
        'critical_districts': True,
        'total_facilities':   True,
    },
    title='Top 20 States by Average Medical Desert Index',
    labels={
        'avg_desert_index': 'Avg Desert Index',
        'state_norm':       'State',
        'desert_pct':       '% Desert Districts',
    },
)
fig_state.update_layout(
    height=550, title_x=0.5, showlegend=False,
    margin=dict(l=10, r=20, t=50, b=10),
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    coloraxis_showscale=False,
    xaxis=dict(gridcolor='rgba(128,128,128,0.2)'),
    yaxis=dict(gridcolor='rgba(0,0,0,0)'),
)
displayHTML(fig_state.to_html(full_html=False, include_plotlyjs='cdn'))

# ── Scatter: risk score vs facility count ─────────────────────
df_scatter = df_map.dropna(subset=['medical_desert_index']).copy()
df_scatter['facilities_capped'] = df_scatter['verified_facilities'].clip(upper=500)

TIER_COLORS = {
    'desert':      '#b91c1c',
    'underserved': '#ef6c00',
    'partial':     '#ffd166',
    'served':      '#2d6a4f',
    'unscored':    '#9ca3af',
}

fig_scatter = px.scatter(
    df_scatter,
    x='facilities_capped',
    y='district_risk_score',
    color='desert_tier',
    color_discrete_map=TIER_COLORS,
    size='medical_desert_index',
    size_max=18,
    hover_name='district_norm',
    hover_data={
        'state_norm':            True,
        'district_risk_score':   ':.1f',
        'verified_facilities':   True,
        'medical_desert_index':  ':.1f',
        'dominant_domain':       True,
        'facilities_capped':     False,
    },
    title='District Risk Score vs Facility Availability',
    labels={
        'facilities_capped':   'Verified Facilities (capped at 500)',
        'district_risk_score': 'Health Risk Score (0–100)',
        'desert_tier':         'Desert Tier',
    },
)
# Annotate medical desert quadrant (top-left: high risk, few facilities)
fig_scatter.add_shape(
    type='rect', x0=0, x1=20, y0=55, y1=100,
    fillcolor='rgba(185,28,28,0.08)',
    line=dict(color='rgba(185,28,28,0.3)', width=1, dash='dot'),
)
fig_scatter.add_annotation(
    x=10, y=97, text='⚠ Medical desert zone',
    showarrow=False, font=dict(size=11, color='#b91c1c'),
)
fig_scatter.update_layout(
    height=520, title_x=0.5,
    margin=dict(l=10, r=10, t=50, b=10),
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    xaxis=dict(gridcolor='rgba(128,128,128,0.15)'),
    yaxis=dict(gridcolor='rgba(128,128,128,0.15)'),
    legend=dict(title='Desert tier', orientation='h', y=-0.15),
)
displayHTML(fig_scatter.to_html(full_html=False, include_plotlyjs='cdn'))

# ── Top 20 deserts table ──────────────────────────────────────
df_top20 = (
    df_map[df_map['desert_tier'].isin(['desert', 'underserved'])]
    [[
        'district_norm', 'state_norm',
        'medical_desert_index', 'desert_tier',
        'district_risk_score', 'risk_tier',
        'dominant_domain',
        'verified_facilities', 'hospital_count',
        'maternal_score', 'child_score', 'access_score',
    ]]
    .sort_values('medical_desert_index', ascending=False)
    .head(20)
    .rename(columns={
        'district_norm':        'District',
        'state_norm':           'State',
        'medical_desert_index': 'Desert Index',
        'desert_tier':          'Desert Tier',
        'district_risk_score':  'Risk Score',
        'risk_tier':            'Risk Tier',
        'dominant_domain':      'Primary Gap',
        'verified_facilities':  'Facilities',
        'hospital_count':       'Hospitals',
        'maternal_score':       'Maternal',
        'child_score':          'Child',
        'access_score':         'Access',
    })
    .round(1)
    .reset_index(drop=True)
)
display(spark.createDataFrame(df_top20))


# # ── CELL 6 of 6: Save HTML files to DBFS ────────────────────────
# # Download via: Databricks UI → Data → DBFS → FileStore → benefitbridge → maps

# EXPORT_DIR = '/dbfs/FileStore/benefitbridge/maps'
# os.makedirs(EXPORT_DIR, exist_ok=True)

# if GEOJSON_OK:
#     fig_district.write_html(f'{EXPORT_DIR}/district_desert_map.html')
#     print(f'Saved → {EXPORT_DIR}/district_desert_map.html')

# fig_state.write_html(f'{EXPORT_DIR}/state_desert_chart.html')
# fig_scatter.write_html(f'{EXPORT_DIR}/risk_vs_facilities_scatter.html')

# print(f'Saved → {EXPORT_DIR}/state_desert_chart.html')
# print(f'Saved → {EXPORT_DIR}/risk_vs_facilities_scatter.html')

# ── Streamlit embed (paste into your app) ────────────────────────
# import streamlit.components.v1 as components
# with open('district_desert_map.html') as f:
#     components.html(f.read(), height=650, scrolling=False)

Bar chart — Chart 1: The longer and redder the bar, the worse the medical desert situation. Ladakh, Nagaland, and Tripura topping the list makes complete geographic sense — they are remote, have very few healthcare facilities in the dataset, and their NFHS health burden scores are high. The index formula amplifies this because low facility count drives the divisor close to 1, so the full risk score carries through. Bihar and Jharkhand at positions 5–6 is the more important policy finding — these are large, densely populated states, not remote territories, meaning millions of people live in underserved districts.
Scatter chart — Chart 2: The dense cluster of dots near x=0 shows that the vast majority of Indian districts have fewer than 20 verified facilities in your dataset. The red and orange dots in the top-left corner are your highest-priority medical deserts — high health burden, almost no facilities. The green dots scattered to the right (x=300–400) are the well-served metros like Mumbai, Pune, and Bengaluru — lots of facilities and relatively lower risk scores. The interesting cases for judges are the dots that sit high on the Y axis but moderately right on X — these are districts where facilities exist but health outcomes are still poor, suggesting a quality or access gap rather than a supply gap.

## Output tables created

| Table | Rows | Description |
|---|---|---|
| `trusted.district_risk_scores` | 706 | Risk score + 5 domain sub-scores per district |
| `trusted.medical_desert_index` | 706 | Risk × inverse facility density — the desert map |

## Join keys for downstream use

Both tables join on `(district_norm, state_norm)` to:
- `trusted.facilities_enriched` — for referral ranking
- `trusted.district_health_context` — for full indicator drill-down
- `trusted.facility_trust_scores` — for combined risk + trust view

## Dashboard queries (Component 8)

```sql
-- Top 20 medical deserts
SELECT district_norm, state_norm, district_risk_score,
       verified_facilities, medical_desert_index, desert_tier
FROM benefits_navigator.trusted.medical_desert_index
WHERE desert_tier = 'desert'
ORDER BY medical_desert_index DESC LIMIT 20;

-- Risk by state (aggregate for map)
SELECT state_norm,
       ROUND(AVG(district_risk_score),1) AS avg_risk,
       COUNT(*) AS districts,
       SUM(CAST(facility_gap_flag AS INT)) AS gap_districts
FROM benefits_navigator.trusted.medical_desert_index
GROUP BY state_norm ORDER BY avg_risk DESC;

-- Dominant domain breakdown (what's driving risk nationally?)
SELECT dominant_domain, COUNT(*) AS districts,
       ROUND(AVG(district_risk_score),1) AS avg_risk
FROM benefits_navigator.trusted.district_risk_scores
GROUP BY dominant_domain ORDER BY districts DESC;
```
